# Posterior parameter recovery

Set `ARTIFACT_DIR`, choose how many recovery cases to run, and draw posterior samples for each case.


In [ ]:
import copy
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from tqdm.auto import tqdm

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

project_root = Path.cwd()
if not (project_root / "tasks").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    import scienceplots  # noqa: F401
except ModuleNotFoundError:
    pass
else:
    plt.style.use("science")

from network_creation.model_inversion import (
    category_probabilities,
    decode_discrete_theta_samples,
    load_posterior_artifact,
)
from network_creation.model_inversion.diagnostics import (
    DiagnosticData,
    load_diagnostic_data,
    marginal_sbc_ranks,
    plot_expected_coverage,
)
from network_creation.model_inversion.recovery import (
    observation_from_policy_outputs,
    parameter_recovery_summary,
    plot_parameter_marginals,
    plot_parameter_pairs,
    plot_posterior_ridgelines,
)
from tasks.policies import prepare_policy_run
import inspect
from tqdm.auto import tqdm

## Config


In [ ]:
ARTIFACT_DIR = "mnpe-20260522T230422Z"

N_RECOVERY_CASES = 1
N_POSTERIOR_SAMPLES_PER_CASE = 1024
SEED = 42
DEVICE = "cpu"
MAX_SAMPLING_TIME_SECONDS = 300

## Helpers


In [ ]:
def resolve_existing_path(raw_path):
    path = Path(raw_path).expanduser()
    if path.is_absolute():
        return path.resolve()
    candidates = [
        project_root / path,
        project_root / "artifacts" / "model_inversion" / path,
        path,
    ]
    if "artifact_dir" in globals():
        candidates.append(artifact_dir / path)
    return next(
        (candidate.resolve() for candidate in candidates if candidate.exists()),
        (project_root / path).resolve(),
    )


def n_agents_values(raw_n_agents):
    if isinstance(raw_n_agents, dict):
        if raw_n_agents.get("kind") != "sampled_discrete_range":
            raise ValueError(f"Unsupported n_agents spec: {raw_n_agents}")
        return list(
            range(
                int(raw_n_agents["min"]),
                int(raw_n_agents["max"]) + 1,
                int(raw_n_agents.get("step", 1)),
            )
        )
    if isinstance(raw_n_agents, list):
        return [int(value) for value in raw_n_agents]
    return [int(raw_n_agents)]


def make_recovery_config(source_config, seed, n_episodes=None):
    config = copy.deepcopy(source_config)
    rng = np.random.default_rng(seed)
    population = config["population"]
    proportions = population["policy_proportions"]

    population["n_agents"] = int(rng.choice(n_agents_values(population["n_agents"])))
    proportions["values"] = [float(rng.choice(proportions["values"]))]
    config["sampling"]["log2_budget"] = 0
    config["sampling"]["seed"] = int(seed)
    config["simulator"]["seed"] = (
        int(source_config["simulator"]["seed"]) + 10000 + int(seed)
    )
    if n_episodes is not None:
        config["simulator"]["n_episodes"] = int(n_episodes)

    return config, {
        "seed": int(seed),
        "n_agents": population["n_agents"],
        "policy_proportion": proportions["values"][0],
        "simulator_seed": config["simulator"]["seed"],
        "n_episodes": config["simulator"]["n_episodes"],
    }


def true_theta_from_prepared(prepared, theta_order):
    input_coordinates = np.asarray(prepared.result_arrays["input_coordinates"])
    coordinates = tuple(int(value) for value in input_coordinates[0])
    names = list(prepared.result_attrs["parameter_names"])
    values = np.asarray(prepared.result_arrays["parameters"])[coordinates]
    by_name = {name: float(value) for name, value in zip(names, values, strict=True)}
    by_name["n_agents"] = float(
        np.asarray(prepared.result_arrays["n_agents"])[coordinates[0]]
    )
    missing = [name for name in theta_order if name not in by_name]
    if missing:
        raise KeyError(f"Theta names missing from recovery config: {missing}")
    return np.asarray([by_name[name] for name in theta_order], dtype=np.float32)


def sample_posterior(artifact, x, n_samples, seed, progress=None):
    chunk_size = 128
    chunks = []
    remaining = int(n_samples)
    chunk_index = 0
    while remaining > 0:
        this_n = min(chunk_size, remaining)
        samples, _ = artifact.sample(
            x,
            num_samples=this_n,
            seed=int(seed) + chunk_index,
            device=DEVICE,
            max_sampling_time=MAX_SAMPLING_TIME_SECONDS,
            return_partial_on_timeout=True,
            show_progress_bars=False,
            max_sampling_batch_size=max(1, 100_000 // this_n),
        )
        if samples.shape[0] == 0:
            raise RuntimeError(
                "Posterior sampling returned no samples before max_sampling_time."
            )
        chunks.append(samples)
        remaining -= int(samples.shape[0])
        if progress is not None:
            progress.update(int(samples.shape[0]))
        chunk_index += 1
    return np.concatenate(chunks, axis=0)[:n_samples].astype(np.float32)

## Load Artifact


In [ ]:
artifact_dir = resolve_existing_path(ARTIFACT_DIR)
artifact = load_posterior_artifact(artifact_dir)
metadata = artifact.metadata
dataset_metadata = metadata["dataset"]
prior_samples = artifact.prior_samples
diagnostic_data = load_diagnostic_data(artifact)
theta_names = list(artifact.theta_order)

source_config_path = resolve_existing_path(dataset_metadata["config_path"])
source_config = json.loads(source_config_path.read_text(encoding="utf-8"))

display(
    {
        "artifact_dir": (
            str(artifact_dir.relative_to(project_root))
            if artifact_dir.is_relative_to(project_root)
            else str(artifact_dir)
        ),
        "posterior_type": type(artifact.posterior).__name__,
        "theta_order": theta_names,
        "x_shape": tuple(dataset_metadata["x_event_shape"]),
        "stored_prior_samples": int(prior_samples.shape[0]),
    }
)

## Learning Curve


In [ ]:
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
except ModuleNotFoundError:
    EventAccumulator = None

learning_scalars = {}
if EventAccumulator is not None and artifact.tensorboard_log_dir.is_dir():
    accumulator = EventAccumulator(str(artifact.tensorboard_log_dir))
    accumulator.Reload()
    available_tags = set(accumulator.Tags().get("scalars", []))
    for tag in (
        "training_loss",
        "validation_loss",
        "epochs_trained",
        "best_validation_loss",
    ):
        if tag in available_tags:
            events = accumulator.Scalars(tag)
            learning_scalars[tag] = {
                "step": np.asarray([event.step for event in events]),
                "value": np.asarray([event.value for event in events]),
            }

curve_tags = [
    tag for tag in ("training_loss", "validation_loss") if tag in learning_scalars
]
if curve_tags:
    fig, ax = plt.subplots(figsize=(8, 4))
    for tag in curve_tags:
        curve = learning_scalars[tag]
        ax.plot(curve["step"], curve["value"], label=tag)
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.set_title("SBI training curve")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    plt.show()

if learning_scalars:
    rows = [
        {
            "tag": tag,
            "n_points": len(curve["value"]),
            "first": float(curve["value"][0]),
            "last": float(curve["value"][-1]),
            "min": float(np.min(curve["value"])),
            "max": float(np.max(curve["value"])),
        }
        for tag, curve in learning_scalars.items()
        if len(curve["value"])
    ]
    display(pd.DataFrame(rows) if pd is not None else rows)
elif EventAccumulator is None:
    display(
        Markdown("TensorBoard is not installed, so no learning curve can be loaded.")
    )
else:
    display(
        Markdown(
            f"No TensorBoard scalar logs found under `{artifact.tensorboard_log_dir}`."
        )
    )

## Generate Recovery Cases


In [ ]:
recovery_cases = []
expected_x_shape = tuple(dataset_metadata["x_event_shape"])

for case_index in range(N_RECOVERY_CASES):
    case_seed = SEED + case_index
    recovery_config, choices = make_recovery_config(source_config, case_seed)
    prepared = prepare_policy_run(recovery_config)
    outputs = prepared.run(0)
    recovery_x = observation_from_policy_outputs(
        outputs,
        artifact,
        n_agents=int(choices["n_agents"]),
        n_actors=int(recovery_config["simulator"]["actor_scheduler"]["n_actors"]),
    )
    if tuple(recovery_x.shape) != expected_x_shape:
        raise ValueError(
            f"recovery_x shape {tuple(recovery_x.shape)} does not match expected shape {expected_x_shape}"
        )

    recovery_cases.append(
        {
            "case": case_index,
            "choices": choices,
            "true_theta": true_theta_from_prepared(prepared, theta_names),
            "x": recovery_x,
        }
    )

choice_rows = [{"case": case["case"], **case["choices"]} for case in recovery_cases]
display(pd.DataFrame(choice_rows) if pd is not None else choice_rows)

## Sample Posterior


In [ ]:
posterior_samples = np.asarray(
    [
        sample_posterior(
            artifact,
            case["x"],
            n_samples=N_POSTERIOR_SAMPLES_PER_CASE,
            seed=SEED + int(case["case"]),
        )
        for case in recovery_cases
    ],
    dtype=np.float32,
)

display(
    {
        "requested_shape": (
            N_RECOVERY_CASES,
            N_POSTERIOR_SAMPLES_PER_CASE,
            len(theta_names),
        ),
        "returned_shape": tuple(posterior_samples.shape),
        "requested_posterior_samples_per_case": N_POSTERIOR_SAMPLES_PER_CASE,
        "returned_posterior_samples_per_case": [
            int(samples.shape[0]) for samples in posterior_samples
        ],
    }
)

## Recovery Summary


In [ ]:
summary_rows = []
for case in recovery_cases:
    rows = parameter_recovery_summary(
        artifact,
        true_theta=case["true_theta"],
        posterior_samples=posterior_samples[int(case["case"])],
    )
    summary_rows.extend({"case": case["case"], **row} for row in rows)

display(pd.DataFrame(summary_rows) if pd is not None else summary_rows)

## Plots


In [ ]:
CASE_TO_PLOT = 0

case = recovery_cases[CASE_TO_PLOT]
case_samples = posterior_samples[CASE_TO_PLOT]
true_theta = case["true_theta"]

fig = plot_parameter_marginals(
    artifact,
    prior_samples=prior_samples,
    posterior_samples=case_samples,
    true_theta=true_theta,
)
fig.set_size_inches(4 * len(theta_names), 4)
plt.show()

if len(theta_names) < 2:
    display(Markdown("Pair plot requires at least two theta dimensions."))
else:
    fig = plot_parameter_pairs(
        artifact,
        prior_samples=prior_samples,
        posterior_samples=case_samples,
        true_theta=true_theta,
        seed=SEED,
    )
    fig.set_size_inches(3 * len(theta_names), 3 * len(theta_names))
    plt.show()

## Optional Diagnostics


In [ ]:
N_DIAGNOSTIC_CASES = 100
N_DIAGNOSTIC_POSTERIOR_SAMPLES_PER_CASE = 2**8

rng = np.random.default_rng(SEED)
requested_cases = N_DIAGNOSTIC_CASES
n_cases = min(N_DIAGNOSTIC_CASES, diagnostic_data.theta.shape[0])
n_agents_column = list(metadata["dataset"]["x_names"]).index("n_agents")
diagnostic_n_agents = (
    diagnostic_data.x[:, 0, n_agents_column]
    if diagnostic_data.x.ndim == 3
    else diagnostic_data.x[:, n_agents_column]
)
n_agent_values, n_agent_groups = np.unique(diagnostic_n_agents, return_inverse=True)
available_counts = np.bincount(n_agent_groups)
target = available_counts * n_cases / diagnostic_data.theta.shape[0]
target_counts = np.floor(target).astype(int)
remainder = n_cases - int(target_counts.sum())
if remainder:
    target_counts[np.argsort(-(target - target_counts))[:remainder]] += 1
diagnostic_indices = np.concatenate(
    [
        rng.choice(np.flatnonzero(n_agent_groups == group), size=count, replace=False)
        for group, count in enumerate(target_counts)
        if count
    ]
).astype(np.int64)
rng.shuffle(diagnostic_indices)
diagnostic_subset = DiagnosticData(
    theta=decode_discrete_theta_samples(
        diagnostic_data.theta[diagnostic_indices], metadata
    ),
    x=diagnostic_data.x[diagnostic_indices],
    source_sample=diagnostic_data.source_sample[diagnostic_indices],
)
diagnostic_posterior_samples = []
progress_total = n_cases * N_DIAGNOSTIC_POSTERIOR_SAMPLES_PER_CASE
sample_posterior_accepts_progress = (
    "progress" in inspect.signature(sample_posterior).parameters
)
with tqdm(
    total=progress_total, desc="Sampling diagnostic posteriors", unit="sample"
) as progress:
    for offset, x in enumerate(diagnostic_subset.x):
        progress.set_postfix(case=f"{offset + 1}/{n_cases}")
        sample_kwargs = {
            "n_samples": N_DIAGNOSTIC_POSTERIOR_SAMPLES_PER_CASE,
            "seed": SEED + offset,
        }
        if sample_posterior_accepts_progress:
            sample_kwargs["progress"] = progress
        samples = sample_posterior(artifact, x, **sample_kwargs)
        if not sample_posterior_accepts_progress:
            progress.update(int(samples.shape[0]))
        diagnostic_posterior_samples.append(samples)
diagnostic_posterior_samples = np.asarray(
    diagnostic_posterior_samples, dtype=np.float32
)

display(
    {
        "diagnostic_cases": int(diagnostic_posterior_samples.shape[0]),
        "posterior_samples_per_case": int(diagnostic_posterior_samples.shape[1]),
        "theta_dim": int(diagnostic_posterior_samples.shape[2]),
    }
)

In [ ]:
def sbc_cdf_num_bins(n_cases, minimum=5):
    n_cases = int(n_cases)
    if n_cases <= 0:
        return 1
    return min(n_cases, max(minimum, n_cases // 20))


ranks = marginal_sbc_ranks(
    artifact,
    diagnostic_posterior_samples,
    diagnostic_subset.theta,
    seed=SEED,
)
coverage = {
    "ranks": ranks,
    "dap_samples": diagnostic_posterior_samples[:, 0, :],
    "num_posterior_samples": N_DIAGNOSTIC_POSTERIOR_SAMPLES_PER_CASE,
    "n_diagnostic_simulations": int(diagnostic_subset.theta.shape[0]),
}
fig, ax = plot_expected_coverage(
    artifact,
    coverage,
    num_bins=sbc_cdf_num_bins(coverage["n_diagnostic_simulations"]),
)
plt.show()

In [ ]:
ridgeline_count = 30
fig = plot_posterior_ridgelines(
    artifact,
    prior_samples=prior_samples,
    posterior_samples=diagnostic_posterior_samples[:ridgeline_count],
    true_theta=diagnostic_subset.theta[:ridgeline_count],
)
plt.show()

In [ ]:
selected_n_agents = diagnostic_n_agents[diagnostic_indices]
for n_agents_value in n_agent_values:
    n_agents_mask = selected_n_agents == n_agents_value
    n_group_cases = int(n_agents_mask.sum())
    if not n_group_cases:
        continue

    n_agents_label = f"{n_agents_value:g}"
    display(Markdown(f"### n_agents = {n_agents_label} ({n_group_cases} cases)"))
    group_coverage = {
        "ranks": ranks[n_agents_mask],
        "dap_samples": diagnostic_posterior_samples[n_agents_mask, 0, :],
        "num_posterior_samples": N_DIAGNOSTIC_POSTERIOR_SAMPLES_PER_CASE,
        "n_diagnostic_simulations": n_group_cases,
    }
    fig, ax = plot_expected_coverage(
        artifact,
        group_coverage,
        num_bins=sbc_cdf_num_bins(n_group_cases),
    )
    fig.suptitle(f"n_agents = {n_agents_label}", y=1.02)
    plt.show()

    group_ridgeline_count = min(ridgeline_count, n_group_cases)
    fig = plot_posterior_ridgelines(
        artifact,
        prior_samples=prior_samples,
        posterior_samples=diagnostic_posterior_samples[n_agents_mask][
            :group_ridgeline_count
        ],
        true_theta=diagnostic_subset.theta[n_agents_mask][:group_ridgeline_count],
    )
    fig.suptitle(f"n_agents = {n_agents_label}", y=1.02)
    plt.show()

## Posterior MAP vs Truth


In [ ]:
MAP_HISTOGRAM_BINS = 40
SCATTER_MARKERS = ("o", "s", "^", "D", "P", "X", "v", "<", ">", "h")


def marginal_posterior_map_estimates(artifact, posterior_samples, bins=MAP_HISTOGRAM_BINS):
    samples = np.asarray(posterior_samples, dtype=np.float32)
    maps = np.full((samples.shape[0], samples.shape[2]), np.nan, dtype=np.float32)

    for theta_index, name in enumerate(artifact.theta_order):
        if artifact.is_discrete_theta(name):
            categories = artifact.category_values(name).astype(np.float32)
            for case_index, values in enumerate(samples[:, :, theta_index]):
                probabilities = category_probabilities(values, categories)
                maps[case_index, theta_index] = categories[int(np.argmax(probabilities))]
            continue

        for case_index, values in enumerate(samples[:, :, theta_index]):
            finite = values[np.isfinite(values)]
            if finite.size == 0:
                continue
            if np.allclose(finite, finite[0]):
                maps[case_index, theta_index] = finite[0]
                continue
            density, edges = np.histogram(finite, bins=bins, density=True)
            mode_bin = int(np.argmax(density))
            maps[case_index, theta_index] = 0.5 * (edges[mode_bin] + edges[mode_bin + 1])

    return maps


def pearson_by_n_agents_label(x, y, selected_n_agents, n_agent_values):
    labels = []
    finite = np.isfinite(x) & np.isfinite(y)
    for n_agents_value in n_agent_values:
        group_mask = finite & (selected_n_agents == n_agents_value)
        if group_mask.sum() < 2 or np.isclose(np.std(x[group_mask]), 0) or np.isclose(np.std(y[group_mask]), 0):
            value = "n/a"
        else:
            value = f"{np.corrcoef(x[group_mask], y[group_mask])[0, 1]:.2f}"
        labels.append(f"n = {n_agents_value:g}: r = {value}")
    return "\n".join(labels)


def categorical_confusion_rates(true_values, map_values, categories):
    counts = np.zeros((len(categories), len(categories)), dtype=int)
    for true_value, map_value in zip(true_values, map_values, strict=True):
        if not np.isfinite(true_value) or not np.isfinite(map_value):
            continue
        true_match = np.flatnonzero(np.isclose(categories, true_value))
        map_match = np.flatnonzero(np.isclose(categories, map_value))
        if len(true_match) and len(map_match):
            counts[int(map_match[0]), int(true_match[0])] += 1

    column_totals = counts.sum(axis=0, keepdims=True)
    rates = np.divide(
        counts,
        column_totals,
        out=np.zeros_like(counts, dtype=np.float32),
        where=column_totals > 0,
    )
    total = int(counts.sum())
    accuracy = np.trace(counts) / total if total else np.nan
    return rates, accuracy


def plot_categorical_confusion(ax, title, true_values, map_values, categories):
    rates, accuracy = categorical_confusion_rates(true_values, map_values, categories)
    image = ax.imshow(
        rates,
        vmin=0.0,
        vmax=1.0,
        cmap="Blues",
        aspect="equal",
        origin="lower",
    )
    for row in range(rates.shape[0]):
        for col in range(rates.shape[1]):
            rate = rates[row, col]
            text_color = "white" if rate >= 0.5 else "black"
            ax.text(
                col,
                row,
                f"{rate:.2f}",
                ha="center",
                va="center",
                fontsize=7,
                color=text_color,
            )

    accuracy_label = "n/a" if not np.isfinite(accuracy) else f"{accuracy:.2f}"
    labels = [f"{value:g}" for value in categories]
    ax.set_title(f"{title}\nMAP accuracy = {accuracy_label}")
    ax.set_xlabel("true value")
    ax.set_ylabel("category with max probability")
    ax.set_xticks(np.arange(len(categories)), labels=labels)
    ax.set_yticks(np.arange(len(categories)), labels=labels)
    ax.set_box_aspect(1)
    ax.minorticks_off()
    ax.tick_params(which="minor", bottom=False, left=False)
    return image


selected_n_agents = diagnostic_n_agents[diagnostic_indices]
posterior_map = marginal_posterior_map_estimates(
    artifact,
    diagnostic_posterior_samples,
)

continuous_theta_indices = [
    index for index, name in enumerate(theta_names) if not artifact.is_discrete_theta(name)
]
categorical_theta_indices = [
    index for index, name in enumerate(theta_names) if artifact.is_discrete_theta(name)
]
markers = [SCATTER_MARKERS[index % len(SCATTER_MARKERS)] for index in range(len(n_agent_values))]

if continuous_theta_indices:
    n_cols = min(3, len(continuous_theta_indices))
    n_rows = int(np.ceil(len(continuous_theta_indices) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 3.8 * n_rows), squeeze=False)

    for panel_index, theta_index in enumerate(continuous_theta_indices):
        ax = axes.ravel()[panel_index]
        name = theta_names[theta_index]
        true_values = diagnostic_subset.theta[:, theta_index]
        map_values = posterior_map[:, theta_index]
        finite = np.isfinite(true_values) & np.isfinite(map_values)

        for marker, n_agents_value in zip(markers, n_agent_values, strict=True):
            group_mask = finite & (selected_n_agents == n_agents_value)
            if np.any(group_mask):
                ax.scatter(
                    true_values[group_mask],
                    map_values[group_mask],
                    s=28,
                    alpha=0.75,
                    marker=marker,
                    facecolor="C0",
                    edgecolor="black",
                    linewidth=0.4,
                    label=f"n = {n_agents_value:g}",
                )

        if np.any(finite):
            lower = float(np.nanmin([true_values[finite].min(), map_values[finite].min()]))
            upper = float(np.nanmax([true_values[finite].max(), map_values[finite].max()]))
            padding = 0.05 * (upper - lower) if upper > lower else 0.05
            lower -= padding
            upper += padding
            ax.plot([lower, upper], [lower, upper], color="black", linestyle="--", linewidth=1)
            ax.set_xlim(lower, upper)
            ax.set_ylim(lower, upper)

        ax.set_title(name)
        ax.text(
            0.03,
            0.97,
            pearson_by_n_agents_label(true_values, map_values, selected_n_agents, n_agent_values),
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=8,
            bbox={"facecolor": "white", "alpha": 0.75, "edgecolor": "none"},
        )
        ax.set_xlabel("true value")
        ax.set_ylabel("posterior MAP")
        ax.grid(alpha=0.2)

    for ax in axes.ravel()[len(continuous_theta_indices) :]:
        ax.set_axis_off()

    handles_by_label = {}
    for ax in axes.ravel()[: len(continuous_theta_indices)]:
        handles, labels = ax.get_legend_handles_labels()
        for handle, label in zip(handles, labels, strict=True):
            handles_by_label.setdefault(label, handle)
    if handles_by_label:
        fig.legend(list(handles_by_label.values()), list(handles_by_label), loc="upper center", ncol=len(handles_by_label), frameon=False)
        fig.subplots_adjust(top=0.78 if n_rows == 1 else 0.88)
    else:
        fig.tight_layout()
    plt.show()

for theta_index in categorical_theta_indices:
    name = theta_names[theta_index]
    categories = artifact.category_values(name).astype(np.float32)
    true_values = diagnostic_subset.theta[:, theta_index]
    map_values = posterior_map[:, theta_index]
    n_cols = min(3, len(n_agent_values))
    n_rows = int(np.ceil(len(n_agent_values) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.8 * n_cols, 3.8 * n_rows), squeeze=False)
    image = None

    for panel_index, n_agents_value in enumerate(n_agent_values):
        ax = axes.ravel()[panel_index]
        group_mask = selected_n_agents == n_agents_value
        image = plot_categorical_confusion(
            ax,
            f"n = {n_agents_value:g}",
            true_values[group_mask],
            map_values[group_mask],
            categories,
        )
        if panel_index % n_cols:
            ax.set_ylabel("")

    for ax in axes.ravel()[len(n_agent_values) :]:
        ax.set_axis_off()

    fig.suptitle(f"{name}: posterior MAP confusion by group size")
    fig.subplots_adjust(top=0.82 if n_rows == 1 else 0.88, right=0.88, wspace=0.35, hspace=0.45)
    if image is not None:
        colorbar_ax = fig.add_axes([0.91, 0.16, 0.018, 0.62 if n_rows == 1 else 0.72])
        colorbar = fig.colorbar(image, cax=colorbar_ax, label="proportion")
        colorbar.ax.minorticks_off()
    plt.show()